# Event Labeling

## Problem Definition

**Question.** How can observed news be aligned to tradable bars and labeled without putting future outcomes into features?

**Role in the workflow.** Create the single event table used by weighting, modeling, and backtesting.

**Inputs.** Local dollar bars, fractional log-price, technical indicators, and timestamped sentiment scores.

**Outputs.** `data/research_data/events/aapl_news_primary_model_2025-01-01_2025-12-31.parquet`.

**Why this method.** News is aligned to the first completed bar at or after publication, then volatility-scaled triple barriers define outcomes.

**Assumptions.** The event start is the feature-observation time; event end, barrier, target, return, and label are metadata/outcomes and never features.

**Handoff.** The labeled event Parquet to `event_weights.ipynb` and the modeling notebooks.


## Real Data Check

Only completed-bar values at `event_start` are joined as features. The final 20% holdout is not explored here; it is isolated in `cross_validation.ipynb` immediately after deterministic event weights are created.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing.event_labeling import (
    drop_labels,
    get_bins,
    get_daily_volatility,
    get_events,
    get_vertical_barriers,
)

market_feature_dir = PROJECT_ROOT / "data/research_data/market/features"
alternative_feature_dir = PROJECT_ROOT / "data/research_data/alternative/features"
event_dir = PROJECT_ROOT / "data/research_data/events"
period = "2025-01-01_2025-12-31"

dollar_bars = pd.read_parquet(market_feature_dir / f"aapl_dollar_bar_{period}.parquet").sort_values("end")
dollar_bars = dollar_bars.drop_duplicates("end", keep="last")
fractional = pd.read_parquet(market_feature_dir / f"aapl_dollar_bar_fractional_{period}.parquet").sort_values("end").drop_duplicates("end", keep="last")
technical = pd.read_parquet(market_feature_dir / f"aapl_dollar_bar_technical_{period}.parquet").sort_values("end").drop_duplicates("end", keep="last")
sentiment = pd.read_parquet(alternative_feature_dir / f"aapl_finbert_sentiment_scores_{period}.parquet").sort_values("created_at")

source_summary = pd.Series(
    {
        "dollar_bars": len(dollar_bars),
        "fractional_features": len(fractional),
        "technical_features": len(technical),
        "sentiment_articles": len(sentiment),
    },
    name="rows",
)
display(source_summary.to_frame())


## Development Preprocessing: News Alignment and Triple Barriers

Duplicate articles are removed by observable time and headline. Multiple stories mapped to the same completed bar are aggregated before labels are computed from the subsequent price path.


In [ ]:
deduplicated_news = sentiment.drop_duplicates(["created_at", "headline"], keep="first")
market_end_index = pd.DatetimeIndex(dollar_bars["end"])
positions = market_end_index.searchsorted(deduplicated_news["created_at"])
valid = positions < len(market_end_index)

aligned_news = deduplicated_news.loc[valid, ["id", "created_at", "sentiment_score"]].copy()
aligned_news["event_start"] = market_end_index[positions[valid]]
assert aligned_news["event_start"].ge(aligned_news["created_at"]).all()

news_events = aligned_news.groupby("event_start").agg(
    news_count=("id", "size"),
    mean_sentiment_score=("sentiment_score", "mean"),
)
news_events.index = pd.DatetimeIndex(news_events.index, name="event_start")

close = dollar_bars.set_index("end")["close"].astype(float)
daily_volatility = get_daily_volatility(close_prices=close, span=50)
vertical_barriers = get_vertical_barriers(news_events.index, close, num_bars=50).rename("vertical_barrier")
target_returns = daily_volatility.reindex(news_events.index)
eligible = target_returns.dropna().index.intersection(vertical_barriers.index)
minimum_target = float(target_returns.loc[eligible].quantile(0.25))

events = get_events(
    close_prices=close,
    event_times=eligible,
    barrier_multipliers=[1.0, 1.0],
    target_returns=target_returns,
    minimum_target_return=minimum_target,
    num_threads=1,
    vertical_barriers=vertical_barriers,
)
labels = get_bins(events, close)
directional = drop_labels(
    events.join(vertical_barriers).join(labels),
    minimum_frequency=0.10,
).rename(columns={"realized_return": "raw_return", "label": "direction_label"})
directional["direction_label"] = directional["direction_label"].astype("int8")

display(pd.Series({"news_event_bars": len(news_events), "eligible": len(eligible), "labeled": len(directional), "minimum_target_return": minimum_target}, name="value").to_frame())


## Feature Table and Output

The feature allowlist contains sentiment, fractional log-price, and 51 technical indicators. Outcomes and identifiers remain present only so later stages can split, score, and backtest.


In [ ]:
technical_columns = [column for column in technical.columns if column not in {"start", "end", "symbol"}]
feature_columns = ["mean_sentiment_score", "fractionally_differenced_log_close", *technical_columns]

event_features = (
    news_events
    .join(fractional.set_index("end"), how="left")
    .join(technical.set_index("end")[technical_columns], how="left")
)
assert not event_features.loc[directional.index, feature_columns].isna().any().any()

model_data = directional.join(event_features).rename_axis("event_start").reset_index()
model_data.insert(1, "symbol", "AAPL")
metadata_columns = [
    "event_start", "symbol", "event_end", "vertical_barrier", "target_return",
    "raw_return", "direction_label", "news_count",
]
model_data = model_data[metadata_columns + feature_columns].sort_values("event_start", ignore_index=True)

assert model_data["event_start"].is_unique
assert set(model_data["direction_label"]) == {-1, 1}
assert len(feature_columns) == 53

event_dir.mkdir(parents=True, exist_ok=True)
event_path = event_dir / f"aapl_news_primary_model_{period}.parquet"
model_data.to_parquet(event_path, index=False)

display(model_data[metadata_columns + feature_columns[:3]].head())
print(event_path)


## Results, Limitations, and Handoff

Triple-barrier outcomes depend on the selected 50-bar horizon, volatility span, and symmetric multipliers. These were fixed before holdout isolation; poor model or backtest performance will be reported rather than relabeled.

The next notebook receives the leakage-auditable event table. No conclusion in this notebook is evidence of live-trading profitability.
